# Stage 1 Retrieval for the Question Answering Pipeline

**Authors:** Roberto Herrera, Bach Nguyen
**Datasets:** HotPotQA, QASPER

## Overview
This notebook implements Stage 1 of a two stage question answering pipeline. Stage 2 uses QA models from the BERT family, with these models reading at most 512 tokens (roughly 2800 characters). Most contexts in our datasets are longer than that.

Stage 1 addresses this by reducing each context before the QA model runs. The context is split into chunks, the chunks are embedded and stored in a ChromaDB vector store, and the question is embedded with the same model to retrieve the chunks closest to it. The retrieved chunks are what Stage 2 receives.

The notebook covers both datasets. HotPotQA gives ten short Wikipedia paragraphs per question. QASPER gives the full text of a research paper per question.

## 1. Setup

In [2]:
import os
import re
import ast
import json
import time
import random
import pandas as pd
from datetime import datetime
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

HOTPOTQA_PATH = "/root/.cache/kagglehub/datasets/jeromeblanchet/hotpotqa-question-answering-dataset/versions/1/hotpot_train_v1.1.json"
QASPER_DIR = "/root/.cache/kagglehub/datasets/thedevastator/qasper-nlp-questions-and-evidence/versions/2"

SEED = 42
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
random.seed(SEED)

def save_result(name, settings, metrics):
    record = {
        "experiment": name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "seed": SEED,
        "settings": settings,
        "metrics": metrics,
    }
    (RESULTS_DIR / (name + ".json")).write_text(json.dumps(record, indent=2))
    return record

## 2. Datasets
**HotpotQA** (Yang et al., 2018) is built from Wikipedia. In the distractor setting each question comes with ten paragraphs, and the file lists the supporting facts, which are the paragraphs and sentences needed to answer. The training file has 90,447 examples.

**QASPER** (Dasigi et al., 2021) is built from full NLP research papers. Each question comes with the paper text and gold evidence passages written by annotators.  The training file has 888 papers with a mean length of about 24,000 characters, and the longest paper has about 168,000 characters.

In [3]:
from collections import Counter

def parse_cell(s):
    s = re.sub(r",\s*dtype=\w+\)", ")", s)
    s = s.replace("array(", "(")
    return ast.literal_eval(s)

def paper_paragraphs(row):
    ft = parse_cell(row["full_text"])
    return [
        (section, p)
        for section, plist in zip(ft["section_name"], ft["paragraphs"])
        for p in plist
        if p.strip()
    ]

with open(HOTPOTQA_PATH) as f:
    hotpot = json.load(f)

qasper = pd.read_csv(QASPER_DIR + "/train.csv", dtype={"id": str})
paper_lengths = [sum(len(p) for _, p in paper_paragraphs(row)) for _, row in qasper.iterrows()]

print("HotpotQA examples", len(hotpot))

print("QASPER papers", len(qasper))
print(pd.Series(paper_lengths).describe())

HotpotQA examples 90447
QASPER papers 888
count       888.000000
mean      24020.396396
std       14525.773967
min           0.000000
25%       15485.250000
50%       23031.500000
75%       28203.750000
max      167959.000000
dtype: float64


## 3. Retrieval functions
Each function splits the context into chunks with LangChain's recursive character splitter, which cuts at paragraph, sentence and word boundaries before cutting inside a word. The chunks are embedded and stored in ChromaDB, the question is embedded with the same model, and the k nearest chunks are returned.

HotpotQA gives each question its own ten paragraphs, so a temporary vector store is built per question. QASPER gives many questions per paper, so one vector store is built per paper and reused for all of its questions.

In [4]:
from uuid import uuid4


def make_splitter(chunk_size, chunk_overlap):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def retrieve_hotpot(example, embedder, k, chunk_size, chunk_overlap):
    """Return (chunk_texts, paragraph_titles) for the top k chunks of one HotpotQA example."""
    splitter = make_splitter(chunk_size, chunk_overlap)
    docs = [
        Document(page_content=piece, metadata={"title": title})
        for title, sentences in example["context"]
        for piece in splitter.split_text(" ".join(sentences))
    ]
    store = Chroma.from_documents(docs, embedder, collection_name="h_" + uuid4().hex)
    results = store.similarity_search(example["question"], k=k)
    store.delete_collection()
    return [r.page_content for r in results], [r.metadata["title"] for r in results]


def build_qasper_store(row, embedder, chunk_size, chunk_overlap):
    """Chunk and embed one paper. Returns None for papers with no text."""
    splitter = make_splitter(chunk_size, chunk_overlap)
    docs = [
        Document(page_content=piece, metadata={"section": section, "para_id": pid})
        for pid, (section, paragraph) in enumerate(paper_paragraphs(row))
        for piece in splitter.split_text(paragraph)
    ]
    if not docs:
        return None
    return Chroma.from_documents(docs, embedder, collection_name="q_" + uuid4().hex)


def retrieve_qasper(store, question, k):
    """Return (chunk_texts, paragraph_ids) for the top k chunks of one question."""
    results = store.similarity_search(question, k=k)
    return [r.page_content for r in results], [r.metadata["para_id"] for r in results]

## 4. Retrieval metrics
Both datasets label the gold paragraphs for each question, so retrieval is scored before any QA model runs by checking which of those paragraphs it returned.

**Recall** is the fraction of gold paragraphs found among the retrieved chunks.

**Hit rate** is the fraction of questions where retrieval succeeded. For HotpotQA that means all gold paragraphs were retrieved. For QASPER it means at least one was, because annotators give alternative evidence for the same question.

**Context length** is the number of characters retrieved, compared with the 2800 character budget of the QA models.

In [5]:
def clean(text):
    return " ".join(text.lower().split())

# HotpotQA
def hotpot_recall(example, retrieved_titles):
    gold = {title for title, _ in example["supporting_facts"]}
    return len(gold & set(retrieved_titles)) / len(gold)

def hotpot_hit(example, retrieved_titles):
    return hotpot_recall(example, retrieved_titles) == 1.0

def answer_contained(example, chunks):
    answer = clean(example["answer"])
    if answer in ("yes", "no"):
        return None
    return answer in clean(" ".join(chunks))

# QASPER
def gold_paragraph_ids(paragraphs, qas, i):
    evidence = [
        clean(e)
        for annotation in qas["answers"][i]["answer"]
        for e in annotation["evidence"]
        if e.strip() and e != "FLOAT SELECTED"
    ]
    return {pid for pid, (_, p) in enumerate(paragraphs) if any(e in clean(p) for e in evidence)}

def qasper_recall(retrieved_ids, gold_ids):
    return len(gold_ids & set(retrieved_ids)) / len(gold_ids)

def qasper_hit(retrieved_ids, gold_ids):
    return len(gold_ids & set(retrieved_ids)) > 0

# Shared
def context_chars(chunks):
    return sum(len(c) for c in chunks)

## 5. Experiments
Each combination of embedding model, chunk size and overlap is run on a fixed random sample of questions from each dataset. The top 5 chunks are retrieved once per question, and the scores for k of 1, 3 and 5 are computed from the first k chunks. Every combination is saved to a JSON file in the results folder, with its settings, scores and timing.

In [6]:
EMBEDDING_MODELS = ["all-MiniLM-L6-v2", "all-mpnet-base-v2", "all-MiniLM-L12-v2"]
CHUNK_SIZES = [250, 500, 1000]
OVERLAPS = [0, 50, 100, 200]
K_VALUES = [1, 3, 5]
N_HOTPOT = 200
N_PAPERS = 50

rng = random.Random(SEED)
hotpot_sample = rng.sample(hotpot, N_HOTPOT)
papers_with_text = [row for _, row in qasper.iterrows() if paper_paragraphs(row)]
qasper_sample = rng.sample(papers_with_text, N_PAPERS)


def mean(values):
    return sum(values) / len(values) if values else None


def new_scores():
    return {k: {"recall": [], "hit": [], "contained": [], "chars": []} for k in K_VALUES}


def summarize(scores):
    return {f"k{k}": {metric: mean(v) for metric, v in d.items()} for k, d in scores.items()}


def run_hotpot(embedder, chunk_size, overlap):
    scores = new_scores()
    rows = []
    start = time.time()
    for ex in hotpot_sample:
        chunks, titles = retrieve_hotpot(ex, embedder, max(K_VALUES), chunk_size, overlap)
        row = {"id": ex["_id"], "type": ex["type"], "level": ex["level"]}
        for k in K_VALUES:
            scores[k]["recall"].append(hotpot_recall(ex, titles[:k]))
            scores[k]["hit"].append(hotpot_hit(ex, titles[:k]))
            contained = answer_contained(ex, chunks[:k])
            if contained is not None:
                scores[k]["contained"].append(contained)
            scores[k]["chars"].append(context_chars(chunks[:k]))
            row[f"hit_k{k}"] = hotpot_hit(ex, titles[:k])
        rows.append(row)
    timing = {"sec_per_question": (time.time() - start) / len(hotpot_sample)}
    return summarize(scores), timing, rows


def run_qasper(embedder, chunk_size, overlap):
    scores = new_scores()
    rows = []
    index_time = query_time = 0
    for paper in qasper_sample:
        paragraphs = paper_paragraphs(paper)
        qas = parse_cell(paper["qas"])
        t = time.time()
        store = build_qasper_store(paper, embedder, chunk_size, overlap)
        index_time += time.time() - t
        for i, question in enumerate(qas["question"]):
            gold = gold_paragraph_ids(paragraphs, qas, i)
            if not gold:
                continue
            t = time.time()
            chunks, ids = retrieve_qasper(store, question, max(K_VALUES))
            query_time += time.time() - t
            row = {"paper": paper["id"], "question": question, "n_gold": len(gold)}
            for k in K_VALUES:
                scores[k]["recall"].append(qasper_recall(ids[:k], gold))
                scores[k]["hit"].append(qasper_hit(ids[:k], gold))
                scores[k]["chars"].append(context_chars(chunks[:k]))
                row[f"hit_k{k}"] = qasper_hit(ids[:k], gold)
            rows.append(row)
        store.delete_collection()
    timing = {
        "index_sec_per_paper": index_time / len(qasper_sample),
        "query_sec_per_question": query_time / max(len(rows), 1),
        "n_questions": len(rows),
    }
    return summarize(scores), timing, rows

In [7]:
for model_name in EMBEDDING_MODELS:
    embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/" + model_name)
    for chunk_size in CHUNK_SIZES:
        for overlap in OVERLAPS:
            for dataset, run in [("hotpotqa", run_hotpot), ("qasper", run_qasper)]:
                name = f"{dataset}_{model_name}_cs{chunk_size}_ov{overlap}"
                if (RESULTS_DIR / (name + ".json")).exists():
                    print("skipping", name)
                    continue
                metrics, timing, rows = run(embedder, chunk_size, overlap)
                settings = {
                    "dataset": dataset,
                    "embedding_model": model_name,
                    "chunk_size": chunk_size,
                    "overlap": overlap,
                    "k_values": K_VALUES,
                }
                save_result(name, settings, {"scores": metrics, "timing": timing, "rows": rows})
                print(name, "hit@3 =", metrics["k3"]["hit"], timing)

skipping hotpotqa_all-MiniLM-L6-v2_cs250_ov0
skipping qasper_all-MiniLM-L6-v2_cs250_ov0
skipping hotpotqa_all-MiniLM-L6-v2_cs250_ov50
skipping qasper_all-MiniLM-L6-v2_cs250_ov50
skipping hotpotqa_all-MiniLM-L6-v2_cs250_ov100
skipping qasper_all-MiniLM-L6-v2_cs250_ov100
skipping hotpotqa_all-MiniLM-L6-v2_cs250_ov200
skipping qasper_all-MiniLM-L6-v2_cs250_ov200
skipping hotpotqa_all-MiniLM-L6-v2_cs500_ov0
skipping qasper_all-MiniLM-L6-v2_cs500_ov0
skipping hotpotqa_all-MiniLM-L6-v2_cs500_ov50
skipping qasper_all-MiniLM-L6-v2_cs500_ov50
skipping hotpotqa_all-MiniLM-L6-v2_cs500_ov100
skipping qasper_all-MiniLM-L6-v2_cs500_ov100
skipping hotpotqa_all-MiniLM-L6-v2_cs500_ov200
skipping qasper_all-MiniLM-L6-v2_cs500_ov200
skipping hotpotqa_all-MiniLM-L6-v2_cs1000_ov0
skipping qasper_all-MiniLM-L6-v2_cs1000_ov0
skipping hotpotqa_all-MiniLM-L6-v2_cs1000_ov50
skipping qasper_all-MiniLM-L6-v2_cs1000_ov50
skipping hotpotqa_all-MiniLM-L6-v2_cs1000_ov100
skipping qasper_all-MiniLM-L6-v2_cs1000_ov100



KeyboardInterrupt



In [9]:
import glob

rows = []
for path in glob.glob(str(RESULTS_DIR / "*.json")):
    print("reading", path)
    with open(path) as f:
        r = json.load(f)
    s = r["settings"]
    m = r["metrics"]["scores"]["k3"]
    t = r["metrics"]["timing"]
    rows.append({
        "dataset": s["dataset"],
        "embedding_model": s["embedding_model"],
        "chunk_size": s["chunk_size"],
        "overlap": s["overlap"],
        "hit_rate": m["hit"],
        "avg_chars": m["chars"],
        "sec_per_question": t.get("sec_per_question", t.get("query_sec_per_question")),
    })

results_df = pd.DataFrame(rows).sort_values(["dataset", "chunk_size", "overlap"])
results_df

reading results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov0.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov100.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov200.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov50.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs250_ov0.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs250_ov100.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs250_ov200.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs250_ov50.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs500_ov0.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs500_ov100.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs500_ov200.json
reading results/hotpotqa_all-MiniLM-L6-v2_cs500_ov50.json
reading results/hotpotqa_all-mpnet-base-v2_cs250_ov0.json
reading results/hotpotqa_all-mpnet-base-v2_cs250_ov100.json
reading results/hotpotqa_all-mpnet-base-v2_cs250_ov50.json
reading results/hotpotqa_all-mpnet-base-v2_cs500_ov100.json
reading results/qasper_all-MiniLM-L6-v2_cs1000_ov0.json
read

,dataset,embedding_model,chunk_size,overlap,hit_rate,avg_chars,sec_per_question
4,hotpotqa,all-MiniLM-L6-v2,250,0,0.355000,634.970000,0.518134
12,hotpotqa,all-mpnet-base-v2,250,0,0.395000,631.595000,3.399000
7,hotpotqa,all-MiniLM-L6-v2,250,50,0.345000,635.835000,0.579536
14,hotpotqa,all-mpnet-base-v2,250,50,0.370000,635.420000,121.855265
5,hotpotqa,all-MiniLM-L6-v2,250,100,0.315000,654.995000,0.637320
13,hotpotqa,all-mpnet-base-v2,250,100,0.355000,660.380000,3.552118
6,hotpotqa,all-MiniLM-L6-v2,250,200,0.200000,707.340000,1.247128
8,hotpotqa,all-MiniLM-L6-v2,500,0,0.410000,1009.455000,0.571775
11,hotpotqa,all-MiniLM-L6-v2,500,50,0.400000,1003.740000,0.573813
9,hotpotqa,all-MiniLM-L6-v2,500,100,0.395000,1032.710000,0.551050


In [10]:
for path in glob.glob(str(RESULTS_DIR / "*cs1000*.json")):
    with open(path) as f:
        r = json.load(f)
    chars = [row.get("hit_k3") is not None and row for row in r["metrics"]["rows"]]
    # actual char counts aren't stored per row yet, only the average
    print(path, r["metrics"]["scores"]["k3"]["chars"])

results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov0.json 1327.42
results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov100.json 1325.655
results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov200.json 1330.365
results/hotpotqa_all-MiniLM-L6-v2_cs1000_ov50.json 1316.635
results/qasper_all-MiniLM-L6-v2_cs1000_ov0.json 1342.7981651376147
results/qasper_all-MiniLM-L6-v2_cs1000_ov100.json 1338.6238532110092
results/qasper_all-MiniLM-L6-v2_cs1000_ov200.json 1367.5504587155963
results/qasper_all-MiniLM-L6-v2_cs1000_ov50.json 1327.119266055046


## 6. Final retrieval configuration
Using the full sweep on all-MiniLM-L6-v2, the selection rule is applied in this order. First, drop any configuration whose average retrieved context exceeds the 2800 character budget. Second, among what remains, choose the configuration with the highest average hit rate across both datasets. Third, break ties by lower latency.

mpnet was 6 to 100 times slower per question in initial testing and was only checked at one shared setting, not swept across chunk sizes, so it is reported but not eligible for selection.

In [11]:
eligible = results_df[(results_df["embedding_model"] == "all-MiniLM-L6-v2") & (results_df["avg_chars"] < 2800)]

avg_hit = eligible.groupby(["chunk_size", "overlap"])["hit_rate"].mean().reset_index()
avg_hit = avg_hit.sort_values("hit_rate", ascending=False)
print(avg_hit)

best_chunk_size, best_overlap = avg_hit.iloc[0][["chunk_size", "overlap"]]
print("Chosen chunk size", best_chunk_size, "overlap", best_overlap)

    chunk_size  overlap  hit_rate
8         1000        0  0.517729
11        1000      200  0.515229
10        1000      100  0.508555
9         1000       50  0.506055
7          500      200  0.461881
4          500        0  0.452706
5          500       50  0.443119
6          500      100  0.440619
0          250        0  0.434381
1          250       50  0.383509
2          250      100  0.377683
3          250      200  0.292661
Chosen chunk size 1000.0 overlap 0.0


## 7. Interface to Stage 2
Stage 2 receives the question and the top k retrieved chunks from retrieve_hotpot or retrieve_qasper, called with chunk_size=1000, overlap=0, k=3, and embedder built from sentence-transformers/all-MiniLM-L6-v2.


In [12]:
import os
import re
import ast
import json
import time
import random
import platform
import pandas as pd
from datetime import datetime
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

HOTPOTQA_PATH = "/root/.cache/kagglehub/datasets/jeromeblanchet/hotpotqa-question-answering-dataset/versions/1/hotpot_train_v1.1.json"
QASPER_DIR = "/root/.cache/kagglehub/datasets/thedevastator/qasper-nlp-questions-and-evidence/versions/2"

SEED = 42
random.seed(SEED)

MACHINE_LABEL = "laptop"   # change this when the notebook runs on another machine, for example "bach" or "vm"


def machine_info():
    with open("/proc/meminfo") as f:
        mem_gb = round(int(f.readline().split()[1]) / 1024 / 1024, 1)
    with open("/proc/cpuinfo") as f:
        cpu = next((line.split(":")[1].strip() for line in f if line.startswith("model name")), "unknown")
    return {
        "label": MACHINE_LABEL,
        "cpu_model": cpu,
        "cpu_cores": os.cpu_count(),
        "memory_gb": mem_gb,
        "platform": platform.platform(),
        "python": platform.python_version(),
    }


MACHINE = machine_info()
print(MACHINE)

RESULTS_DIR = Path("results") / MACHINE_LABEL
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save_result(name, settings, metrics):
    record = {
        "experiment": name,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "seed": SEED,
        "machine": MACHINE,
        "settings": settings,
        "metrics": metrics,
    }
    (RESULTS_DIR / (name + ".json")).write_text(json.dumps(record, indent=2))
    return record

{'label': 'laptop', 'cpu_model': '13th Gen Intel(R) Core(TM) i7-1355U', 'cpu_cores': 12, 'memory_gb': 7.6, 'platform': 'Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.35', 'python': '3.10.12'}
